# NeuroGolf submission builder
exp_id: `GOLF_20260608_004b_beicicc_structural_pass_mix`
base dataset: `octaviograu/neurogolf-manual-rewrites-v205` | overlay dataset: `beicicc/neurogolf-6645-39-open-submission-artifact`


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import zipfile

EXP_ID = 'GOLF_20260608_004b_beicicc_structural_pass_mix'
GIT_COMMIT = '0b86e8b'
SOURCE_IDS = ['SRC_KAGGLE_NOTEBOOK_BEICICC_6645']
BASE_INPUT = Path('/kaggle/input/neurogolf-manual-rewrites-v205')
BASE_SOURCE_SUBDIR = 'submission'
OVERLAY_INPUT = Path('/kaggle/input/neurogolf-6645-39-open-submission-artifact')
OVERLAY_SOURCE_SUBDIR = ''
CHANGED_TASKS = ['task001', 'task011', 'task015', 'task026', 'task043', 'task050', 'task052', 'task059', 'task062', 'task063', 'task070', 'task072', 'task073', 'task075', 'task076', 'task083', 'task087', 'task092', 'task095', 'task098', 'task100', 'task108', 'task113', 'task117', 'task132', 'task135', 'task136', 'task140', 'task142', 'task144', 'task146', 'task147', 'task152', 'task166', 'task169', 'task171', 'task179', 'task188', 'task207', 'task210', 'task211', 'task223', 'task229', 'task236', 'task241', 'task245', 'task246', 'task248', 'task250', 'task256', 'task261', 'task266', 'task272', 'task273', 'task276', 'task281', 'task282', 'task291', 'task299', 'task302', 'task309', 'task311', 'task317', 'task318', 'task326', 'task331', 'task335', 'task350', 'task352', 'task359', 'task360', 'task366', 'task371', 'task372', 'task374', 'task388', 'task389', 'task398', 'task400']
WORK = Path('/kaggle/working')
OUT_DIR = WORK / 'submission_files'
OUT_DIR.mkdir(exist_ok=True)
zip_path = WORK / 'submission.zip'

def resolve_source_dir(dataset_input: Path, source_subdir: str) -> Path:
    source_dir = dataset_input / source_subdir if source_subdir else dataset_input
    if source_dir.exists() and any(source_dir.glob('task*.onnx')):
        return source_dir
    candidates = [p for p in dataset_input.rglob('task001.onnx')]
    if candidates:
        return candidates[0].parent
    raise FileNotFoundError(f'No task001.onnx under {dataset_input}')

base_dir = resolve_source_dir(BASE_INPUT, BASE_SOURCE_SUBDIR)
overlay_dir = resolve_source_dir(OVERLAY_INPUT, OVERLAY_SOURCE_SUBDIR)

for src in sorted(base_dir.glob('task*.onnx')):
    shutil.copy2(src, OUT_DIR / src.name)

for task_name in CHANGED_TASKS:
    overlay = overlay_dir / f'{task_name}.onnx'
    if not overlay.exists():
        raise FileNotFoundError(f'Missing overlay task {task_name} in {overlay_dir}')
    shutil.copy2(overlay, OUT_DIR / overlay.name)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src in sorted(OUT_DIR.glob('task*.onnx')):
        zf.write(src, arcname=src.name)

h = hashlib.sha256()
with zip_path.open('rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        h.update(chunk)

manifest = {
    'exp_id': EXP_ID,
    'git_commit': GIT_COMMIT,
    'source_ids': SOURCE_IDS,
    'base_dataset_slug': 'octaviograu/neurogolf-manual-rewrites-v205',
    'overlay_dataset_slug': 'beicicc/neurogolf-6645-39-open-submission-artifact',
    'base_dir': str(base_dir),
    'overlay_dir': str(overlay_dir),
    'changed_tasks': CHANGED_TASKS,
    'package_sha256': h.hexdigest(),
    'file_count': len(list(OUT_DIR.glob('task*.onnx'))),
    'package_size': zip_path.stat().st_size,
}
print(json.dumps(manifest, indent=2))
print('submission.zip is ready at', zip_path)
